# Lab 2: Dense Variational Autoencoder for Faces

| | |
|---|---|
| **Name** | Anant Arya |
| **Roll No** | A24ARIU0014 |
| **Dataset** | LFW (deep-funneled), 45 × 45 RGB crops |

**Goal:** A normal autoencoder turns each face into one fixed point. A VAE turns each face into a small *cloud* (a Gaussian with mean μ and spread σ). During training we pick a random point from that cloud and ask the decoder to rebuild the face from it. The KL term keeps all the clouds close to N(0, I), so afterwards we can sample from N(0, I) and get new faces.

## 1. Environment and LFW data

In [ ]:
# Kaggle images normally include these packages. For a minimal local environment:
# %pip install torch torchvision matplotlib pandas scikit-learn pillow

In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 45
CHANNELS = 3
INPUT_DIM = CHANNELS * IMAGE_SIZE * IMAGE_SIZE
LATENT_DIM = 100
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3
BETA = 1.0
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

# Change only this line for a local setup if LFW_DATASET_PATH is not set.
LOCAL_DATASET_PATH = Path("data/lfw-deepfunneled")

print(f"Running on {DEVICE}")

In [ ]:
def resolve_lfw_root():
    env_path = os.environ.get("LFW_DATASET_PATH")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    if Path("/kaggle/input").exists():
        candidates.extend([
            Path("/kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"),
            Path("/kaggle/input/lfw-dataset"),
        ])
    candidates.extend([
        LOCAL_DATASET_PATH.expanduser(),
        Path("lfw-deepfunneled"),
    ])

    for root in candidates:
        if root.exists() and next(root.rglob("*.jpg"), None) is not None:
            return root
    checked = "\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(
        "No LFW .jpg files were found. Attach lfw-dataset on Kaggle, set "
        "LFW_DATASET_PATH, or edit LOCAL_DATASET_PATH. Checked:\n" + checked
    )


DATASET_ROOT = resolve_lfw_root()
image_paths = sorted(DATASET_ROOT.rglob("*.jpg"))
image_df = pd.DataFrame({
    "path": image_paths,
    "person": [path.parent.name for path in image_paths],
})

# Preserve the original notebook's balancing choice: remove identities with 25+
# photographs so a few celebrities do not dominate reconstruction training.
image_df = image_df.groupby("person", group_keys=False).filter(lambda rows: len(rows) < 25)
image_df = image_df.reset_index(drop=True)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Using {len(image_df):,} images from {image_df['person'].nunique():,} people")
image_df.head()

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        path = self.dataframe.loc[index, "path"]
        with Image.open(path) as image:
            image = image.convert("RGB")
            # LFW deep-funneled images are 250x250. This reproduces the original
            # notebook's central crop while remaining safe for other image sizes.
            width, height = image.size
            margin_x = min(80, max(0, (width - 1) // 3))
            margin_y = min(80, max(0, (height - 1) // 3))
            image = image.crop((margin_x, margin_y, width - margin_x, height - margin_y))
            return self.transform(image)


transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),  # pixel values in [0, 1]
])

train_df, val_df = train_test_split(image_df, test_size=0.2, random_state=SEED)
train_dataset = FaceDataset(train_df, transform)
val_dataset = FaceDataset(val_df, transform)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)

faces = next(iter(train_loader))
assert faces.ndim == 4 and faces.shape[1:] == (3, 45, 45)
assert 0 <= faces.min() and faces.max() <= 1

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for face, axis in zip(faces[:10], axes.flat):
    axis.imshow(face.permute(1, 2, 0))
    axis.axis("off")
plt.tight_layout()

In [ ]:
# My helper: show a batch of [N, 3, H, W] images in one row.
def show_row(images, title=None, axes_row=None):
    images = images.detach().cpu().clamp(0, 1)
    if axes_row is None:
        fig, axes_row = plt.subplots(1, len(images), figsize=(1.6 * len(images), 1.9))
    for img, ax in zip(images, axes_row):
        ax.imshow(img.permute(1, 2, 0))
        ax.set_xticks([]); ax.set_yticks([])
    if title:
        axes_row[0].set_ylabel(title, fontsize=9)

print(f"Train images: {len(train_dataset)} | Validation images: {len(val_dataset)}")
print(f"Each image is flattened to {INPUT_DIM} numbers (3 x 45 x 45)")

## 2. From autoencoder to VAE

**Q1. What are the shapes of `mu`, `logvar` and `z`?**
All three are **[N, 100]**: one 100-number vector per image. `mu` is the centre of the Gaussian, `logvar` is the log-variance for each of the 100 dimensions, and `z` is one random sample drawn from that Gaussian. So `z` has to have the same shape.

**Q2. Why must the final `mu` and `logvar` layers have no ReLU?**
ReLU only lets out values ≥ 0. But:
- `mu` must be able to be negative, because the prior N(0, I) is centred at 0, so about half of the latent values should be below zero.
- `logvar` must be able to be negative too. A negative log-variance means variance < 1, i.e. the encoder is fairly sure about a face. With ReLU, logvar ≥ 0 means variance ≥ 1 always, so the encoder could never become confident.

So these two heads are plain `nn.Linear` layers that can output any real number.

**Q3. Why does drawing `z` directly from the encoder's distribution break normal backpropagation?**
Sampling is a random step. If we write `z = sample(N(mu, sigma²))`, then `z` is just a number that came out of a random generator. There is no formula connecting it to `mu` and `sigma`, so there is no derivative ∂z/∂μ or ∂z/∂σ to follow back. The gradient stops at the sampling step, and the encoder heads never learn.

The **reparameterization trick** fixes this by moving the randomness to a separate input: `z = mu + sigma * eps`, with `eps ~ N(0, I)`. Now `eps` is the only random part (and needs no gradient), while `z` is a simple add-and-multiply of `mu` and `sigma`, which we *can* differentiate: ∂z/∂μ = 1 and ∂z/∂σ = ε.

## 3. Implement the probabilistic encoder

**Idea:** The shared encoder ends in a 1000-number feature vector. Two separate linear layers read that same vector: one predicts μ and the other predicts log σ². To sample, compute σ = exp(0.5 · logvar), draw ε from N(0, 1), and return μ + σ · ε.

In [ ]:
class DenseFaceVAE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, latent_dim=LATENT_DIM):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1500),
            nn.ReLU(),
            nn.Linear(1500, 1000),
            nn.ReLU(),
        )

        # TODO 1: two heads on the same 1000 features (no activation on either)
        self.mu_head = nn.Linear(1000, latent_dim)       # mean of q(z|x)
        self.logvar_head = nn.Linear(1000, latent_dim)   # log of the variance of q(z|x)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1000),
            nn.ReLU(),
            nn.Linear(1000, 1500),
            nn.ReLU(),
            nn.Linear(1500, input_dim),
            nn.Sigmoid(),
        )

    def encode(self, x):
        features = self.encoder(x.flatten(start_dim=1))
        return self.mu_head(features), self.logvar_head(features)

    def reparameterize(self, mu, logvar):
        # TODO 2: z = mu + sigma * eps
        sigma = torch.exp(0.5 * logvar)   # logvar = log(sigma^2), so sigma = e^(logvar / 2)
        eps = torch.randn_like(sigma)     # the only random part: N(0, 1), same shape as sigma
        z = mu + sigma * eps              # gradient flows to mu and sigma through this line
        return z

    def decode(self, z):
        pixels = self.decoder(z)
        return pixels.view(-1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

    def forward(self, x):
        # TODO 3: image -> (mu, logvar) -> random z -> rebuilt image
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstruction = self.decode(z)
        return reconstruction, mu, logvar

In [ ]:
# These checks give fast feedback before expensive training.
model = DenseFaceVAE().to(DEVICE)
test_batch = faces[:4].to(DEVICE)
reconstruction, mu, logvar = model(test_batch)

assert reconstruction.shape == test_batch.shape
assert mu.shape == logvar.shape == (4, LATENT_DIM)
assert reconstruction.min() >= 0 and reconstruction.max() <= 1

# Sampling should normally give different z values, while gradients still reach
# both distribution heads.
z1 = model.reparameterize(mu, logvar)
z2 = model.reparameterize(mu, logvar)
assert not torch.equal(z1, z2)
(z1.mean()).backward()
assert model.mu_head.weight.grad is not None
assert model.logvar_head.weight.grad is not None
model.zero_grad(set_to_none=True)
print("Model checks passed.")

In [ ]:
# Shapes for this batch of 4 (answers Q1 in code)
print("input         ", tuple(test_batch.shape))
print("flattened     ", tuple(test_batch.flatten(start_dim=1).shape))
print("mu            ", tuple(mu.shape))
print("logvar        ", tuple(logvar.shape))
print("z             ", tuple(z1.shape))
print("reconstruction", tuple(reconstruction.shape))
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Implement the negative ELBO

**Idea:** loss per image = (how badly we rebuilt the pixels) + β × (how far q(z|x) is from N(0, I)).

- **Reconstruction:** binary cross-entropy summed over all 6075 pixel values.
- **KL:** for a diagonal Gaussian it has a closed form, so no sampling is needed: KL = −½ Σ (1 + logvar − μ² − e^logvar).

Both are summed over the batch and divided by the batch size N, so the numbers are "per image".

In [ ]:
def vae_loss(reconstruction, target, mu, logvar, beta=1.0):
    # TODO 4
    batch_size = target.size(0)

    # Part 1: reconstruction error, summed over every pixel, per image
    recon_loss = F.binary_cross_entropy(reconstruction, target, reduction="sum") / batch_size

    # Part 2: KL( N(mu, sigma^2) || N(0, 1) ), summed over the 100 dims, per image
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size

    total = recon_loss + beta * kl_loss       # keep the graph: we backprop through this
    return total, recon_loss.detach(), kl_loss.detach()

In [ ]:
# At mu=0 and logvar=0, q(z|x) equals the unit-normal prior, so KL must be zero.
dummy_x = torch.full((2, 3, 45, 45), 0.5, device=DEVICE)
dummy_mu = torch.zeros(2, LATENT_DIM, device=DEVICE)
dummy_logvar = torch.zeros_like(dummy_mu)
total, recon_term, kl_term = vae_loss(dummy_x, dummy_x, dummy_mu, dummy_logvar)
assert torch.isclose(kl_term, torch.tensor(0.0, device=DEVICE))
assert total.requires_grad is False  # dummy tensors do not require gradients

# With model outputs, the total must carry a gradient path.
reconstruction, mu, logvar = model(test_batch)
total, _, _ = vae_loss(reconstruction, test_batch, mu, logvar, beta=BETA)
assert total.requires_grad
print("Loss checks passed.")

In [ ]:
# Extra check: compare my KL formula with PyTorch's built-in KL for random mu/logvar
from torch.distributions import Normal, kl_divergence

check_mu = torch.randn(5, LATENT_DIM)
check_logvar = torch.randn(5, LATENT_DIM)
_, _, my_kl = vae_loss(torch.full((5, 3, 45, 45), 0.5), torch.full((5, 3, 45, 45), 0.5),
                       check_mu, check_logvar)
q = Normal(check_mu, torch.exp(0.5 * check_logvar))
p = Normal(torch.zeros_like(check_mu), torch.ones_like(check_mu))
library_kl = kl_divergence(q, p).sum(dim=1).mean()
print(f"my KL = {my_kl.item():.4f} | torch.distributions KL = {library_kl.item():.4f}")
assert torch.isclose(my_kl, library_kl, rtol=1e-4)

## 5. Train the VAE

**Idea:** Exactly the usual PyTorch loop. The only VAE-specific part is that the model returns three things and the loss uses all of them.

In [ ]:
def train_step(model, batch, optimizer, beta):
    # TODO 5
    batch = batch.to(DEVICE, non_blocking=True)                   # 1. data to GPU/CPU
    optimizer.zero_grad()                                         # 2. clear old gradients
    reconstruction, mu, logvar = model(batch)                     # 3. forward pass
    total, recon_loss, kl_loss = vae_loss(reconstruction, batch,  # 4. negative ELBO
                                          mu, logvar, beta)
    total.backward()                                              # 5. backpropagate
    optimizer.step()                                              # 6. update weights
    return total.item(), recon_loss.item(), kl_loss.item()

In [ ]:
@torch.no_grad()
def evaluate(model, loader, beta):
    model.eval()
    totals = np.zeros(3, dtype=np.float64)
    examples = 0
    for batch in loader:
        batch = batch.to(DEVICE, non_blocking=True)
        reconstruction, mu, logvar = model(batch)
        values = vae_loss(reconstruction, batch, mu, logvar, beta)
        batch_size = batch.size(0)
        totals += np.array([value.item() for value in values]) * batch_size
        examples += batch_size
    return totals / examples


model = DenseFaceVAE().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
history = {"train_total": [], "val_total": [], "val_reconstruction": [], "val_kl": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_total = 0.0
    seen = 0
    for batch in train_loader:
        total, reconstruction_term, kl_term = train_step(model, batch, optimizer, BETA)
        running_total += total * batch.size(0)
        seen += batch.size(0)

    val_total, val_reconstruction, val_kl = evaluate(model, val_loader, BETA)
    history["train_total"].append(running_total / seen)
    history["val_total"].append(val_total)
    history["val_reconstruction"].append(val_reconstruction)
    history["val_kl"].append(val_kl)
    print(
        f"Epoch {epoch:02d} | train {running_total / seen:.2f} | "
        f"val {val_total:.2f} | recon {val_reconstruction:.2f} | KL {val_kl:.2f}"
    )

plt.plot(history["train_total"], label="train total")
plt.plot(history["val_total"], label="validation total")
plt.xlabel("Epoch")
plt.ylabel("Negative ELBO per image")
plt.legend()
plt.show()

In [ ]:
# The two parts of the validation loss, plotted separately (they have very different sizes)
epochs = range(1, EPOCHS + 1)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.5))
left.plot(epochs, history["val_reconstruction"], marker="o", color="tab:blue")
left.set_title("Validation reconstruction (BCE per image)")
left.set_xlabel("Epoch")
right.plot(epochs, history["val_kl"], marker="o", color="tab:red")
right.set_title("Validation KL per image")
right.set_xlabel("Epoch")
plt.tight_layout()
plt.show()

**What the curves show:**
- The total loss falls quickly in the first epochs and then flattens. Train and validation stay close, so the model is not badly overfitting.
- Almost all of the total is the reconstruction term, because it is a sum over 6075 pixel values while KL is a sum over only 100 latent dimensions.
- KL usually *rises* early on. At the start the encoder outputs are close to the prior, so KL ≈ 0. As the encoder learns to put different faces in different places, KL grows. Then it settles where the gain in reconstruction balances the KL cost.

## 6. Inspect reconstructions

In [ ]:
model.eval()
validation_faces = next(iter(val_loader))[:8].to(DEVICE)
with torch.no_grad():
    mu, _ = model.encode(validation_faces)
    reconstructed_faces = model.decode(mu)

originals = validation_faces.cpu()
reconstructed_faces = reconstructed_faces.cpu()
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for index in range(8):
    axes[0, index].imshow(originals[index].permute(1, 2, 0))
    axes[1, index].imshow(reconstructed_faces[index].permute(1, 2, 0).clamp(0, 1))
    axes[0, index].axis("off")
    axes[1, index].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Reconstructed")
plt.tight_layout()

**Comparison (original vs reconstruction):**

| Aspect | What I see |
|---|---|
| Identity | General face shape and layout are kept, but faces look more "average". Fine features that make a person recognisable are mostly lost. |
| Pose | Head position and tilt are kept well. They are large, low-detail patterns, which the model learns first. |
| Colour | Overall skin tone, hair colour and background brightness are close to the original. |
| Fine detail | Clearly blurry: eyes, teeth, hair texture and edges are smoothed. The dense layers flatten the image and have no built-in sense of neighbouring pixels, and the KL term also pushes towards smooth outputs. |

## 7. Generate new faces

**Idea:** No input image this time. Draw z straight from the prior N(0, I) and send it to the decoder.

In [ ]:
model.eval()
with torch.no_grad():
    # TODO 6: 10 random points from N(0, I) -> decoder
    z_prior = torch.randn(10, LATENT_DIM, device=DEVICE)
    generated_faces = model.decode(z_prior)

In [ ]:
assert generated_faces.shape == (10, 3, 45, 45)
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for face, axis in zip(generated_faces.cpu(), axes.flat):
    axis.imshow(face.permute(1, 2, 0).clamp(0, 1))
    axis.axis("off")
plt.tight_layout()

**Observation:** The samples are blurry, but they are clearly face-like: a skin-coloured oval in the middle, darker regions for eyes and hair, and a plain background. None of them is a copy of a training image. This works because the KL term trained the encoder to keep its outputs near N(0, I), so random points from N(0, I) land in parts of the latent space the decoder knows how to decode.

## 8. Experiment: what does β control?

I compare **β = 0**, **β = 1** (the model from Section 5) and **β = 4**. For β = 0 and β = 4 I train fresh models, keeping data, epochs, learning rate, batch size and architecture the same. For each model I record:
- validation reconstruction loss and validation KL;
- reconstructions of the same 8 validation faces;
- 8 prior samples, decoded from the **same** random z vectors for every model

In [ ]:
def train_fresh_vae(beta):
    torch.manual_seed(SEED)
    vae = DenseFaceVAE().to(DEVICE)
    opt = torch.optim.Adam(vae.parameters(), lr=LEARNING_RATE)
    for epoch in range(1, EPOCHS + 1):
        vae.train()
        for batch in train_loader:
            train_step(vae, batch, opt, beta)
    return vae


beta_models = {1.0: model}          # reuse the beta = 1 model trained in Section 5
for beta in [0.0, 4.0]:
    print(f"Training fresh model with beta = {beta} ...")
    beta_models[beta] = train_fresh_vae(beta)
beta_models = dict(sorted(beta_models.items()))



print("Done.")

In [ ]:
rows = []
for beta, vae in beta_models.items():
    _, val_recon, val_kl = evaluate(vae, val_loader, beta)
    rows.append({"beta": beta, "val reconstruction (BCE)": round(val_recon, 2), "val KL": round(val_kl, 2)})

results_df = pd.DataFrame(rows).set_index("beta")
results_df

In [ ]:
torch.manual_seed(0)
shared_z = torch.randn(8, LATENT_DIM, device=DEVICE)    # same prior samples for every beta
val_faces = next(iter(val_loader))[:8].to(DEVICE)

fig, axes = plt.subplots(1 + 2 * len(beta_models), 8, figsize=(13, 1.7 * (1 + 2 * len(beta_models))))
show_row(val_faces, "original", axes[0])
row = 1
for beta, vae in beta_models.items():
    vae.eval()
    with torch.no_grad():
        recon = vae.decode(vae.encode(val_faces)[0])
        samples = vae.decode(shared_z)
    show_row(recon, f"recon b={beta:g}", axes[row])
    show_row(samples, f"sample b={beta:g}", axes[row + 1])
    row += 2
plt.tight_layout()
plt.show()

**Experiment table** (numbers come from the table printed above)

| β | Reconstruction | KL | Prior-sample observations |
|---|---|---|---|
| 0 | Lowest BCE: sharpest reconstructions | Very large, since nothing holds it down | Mostly noisy, washed-out or broken faces. Prior z does not match where real faces sit. |
| 1 | Medium BCE: blurry but recognisable | Medium | Blurry but clearly face-like and varied |
| 4 | Highest BCE: very smooth, "average" faces, identity mostly lost | Smallest | Smooth, similar-looking average faces with little variety |

**Q1. What happens at β = 0, and why is that model no longer encouraged to have a sampleable N(0, I) latent space?**
With β = 0 the loss is only reconstruction, so the KL term is switched off. The model is then just an autoencoder with some noise added. Nothing stops the encoder from placing faces wherever it likes: μ values can be large, and logvar can be made very negative so the noise almost vanishes. That gives the best reconstructions and a large KL. But real faces no longer sit around N(0, I), so a random z ~ N(0, I) usually lands in an area the decoder never saw during training, and the "generated" faces look broken.

**Q2. How does increasing β change reconstruction fidelity and prior samples?**
A bigger β charges more for every unit of KL, so the encoder keeps each q(z|x) closer to N(0, I), with more overlap and more noise. Reconstructions get blurrier and lose identity (higher BCE), and KL goes down. Prior samples get "safer": they look more like real faces than at β = 0, but at large β they become smooth, similar, average faces. So β trades **reconstruction quality** against **a well-organised latent space you can sample from**. β = 1 is the standard ELBO and sits in between.

**Q3. Why can a VAE be built without convolutional layers? What is the disadvantage of a dense model for images?**
What makes a model a VAE is the *probabilistic* part: an encoder that outputs μ and logvar, the reparameterization trick, and the reconstruction + KL loss. None of that depends on the layer type. Any network that maps x → (μ, logvar) and z → x̂ works, including plain Linear layers.
The downside is that flattening throws away the 2D layout. A Linear layer treats pixel (10, 10) and pixel (10, 11) as unrelated inputs and has to learn every local pattern separately for every position. That means:
- a huge number of parameters (the first layer alone has 6075 × 1500 ≈ 9 million weights);
- no shared filters, and no built-in tolerance to small shifts;
- blurrier results.

A convolutional VAE reuses small filters across the image, so it needs far fewer parameters and keeps edges and textures much better.

## Optional extension: interpolation between two faces

Encode two validation faces, take their means μ_A and μ_B, and decode points along the straight line between them: z = (1 − t)·μ_A + t·μ_B.

In [ ]:
model.eval()
with torch.no_grad():
    mu_pair, _ = model.encode(val_faces[:2])
    steps = torch.linspace(0, 1, 8, device=DEVICE).unsqueeze(1)   # t = 0, 1/7, ..., 1
    path = (1 - steps) * mu_pair[0] + steps * mu_pair[1]          # [8, LATENT_DIM]
    morph = model.decode(path)

fig, axes = plt.subplots(1, 10, figsize=(15, 1.9))
show_row(val_faces[:1], "face A", axes[:1])
show_row(morph, None, axes[1:9])
show_row(val_faces[1:2], "face B", axes[9:])
axes[1].set_title("t = 0", fontsize=8)
axes[8].set_title("t = 1", fontsize=8)
plt.tight_layout()
plt.show()

The in-between images change smoothly from one face to the other: skin tone, pose and hair change gradually instead of jumping. This shows the VAE latent space is continuous, i.e. nearby z values decode to similar faces, which is exactly what the KL term encourages.